# VisionOps Guard - Google Colab Training Pipeline 🚀
**Objective:** Train YOLOv8/YOLOv11 Safety PPE Detection Model with GPU Acceleration.

> **Step 0:** Ensure GPU runtime is active: `Runtime` -> `Change runtime type` -> select `T4 GPU`.

In [ ]:
# 1. Verify NVIDIA GPU Acceleration
!nvidia-smi
import torch
print(f"[*] PyTorch Version: {torch.__version__}")
print(f"[*] CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[*] Active GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Install Core Dependencies
!pip install --quiet ultralytics onnx onnxruntime albumentations pyyaml

In [ ]:
# 3. Clone Repository or Setup Project Structure
# If working from GitHub:
# !git clone https://github.com/your-username/visionops-guard.git
# %cd visionops-guard

# Setup Colab Dataset Configuration
import yaml

dataset_config = {
    'path': '/content/visionops-guard/data/processed',
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {
        0: 'person',
        1: 'hardhat',
        2: 'vest',
        3: 'no_hardhat',
        4: 'no_vest'
    }
}

with open('/content/dataset_colab.yaml', 'w') as f:
    yaml.dump(dataset_config, f)
print("[+] Colab dataset.yaml generated.")

In [ ]:
# 4. Launch Model Training with Ultralytics YOLO
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # Pre-trained Nano backbone for ultra-fast training & low latency

results = model.train(
    data='/content/dataset_colab.yaml',
    epochs=25,
    imgsz=640,
    batch=16,
    device=0,  # Utilize Colab T4 GPU
    workers=2,
    project='runs/colab_train',
    name='visionops_ppe'
)

In [ ]:
# 5. Export Best Model to ONNX Format (Optimized for Production Serving)
best_weights = 'runs/colab_train/visionops_ppe/weights/best.pt'
trained_model = YOLO(best_weights)
onnx_path = trained_model.export(format='onnx', imgsz=640, dynamic=False, simplify=True)
print(f"[+] Exported ONNX Model to: {onnx_path}")

In [ ]:
# 6. Direct Download Weights to Your Local Machine
from google.colab import files

print("[*] Downloading best.pt and best.onnx to your local computer...")
files.download('runs/colab_train/visionops_ppe/weights/best.pt')
files.download('runs/colab_train/visionops_ppe/weights/best.onnx')